# Lab 7: Word Sequences

Compute the similarity of the *SMTeuroparl* sentence pairs (test set) using words plus named entities (NEs) and the Jaccard coefficient, where each NE is kept as a single item (e.g. `['John Smith', 'is', 'working']`), and compare it with the gold standard using the Pearson correlation.

The NEs are found with three tools: NLTK (`ne_chunk`), spaCy (`en_core_web_sm`) and FreeLing (TextServer). Each one is compared with the words of its own tokenizer without NEs, so the only change is the merging of the NEs.

## Setup

Import the libraries, and download the NLTK packages (tokenizer, POS tagger and NE chunker) and the spaCy model. The TextServer answers of Lab 6 are in `common/`, shared by the labs.

In [1]:
import sys
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
if "google.colab" in sys.modules:
    %pip install -q nltk spacy pandas scipy
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

DATA_DIR = ROOT / "test_gold"
NLTK_DIR = ROOT / "nltk_data"
COMMON_DIR = ROOT / "common"
UKB_CACHE = COMMON_DIR / "ukb_senses.json"

# Notebook libs
import json

import nltk
import pandas as pd
import spacy
from nltk.metrics.distance import jaccard_distance
from scipy.stats import pearsonr

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR)
nltk.download("averaged_perceptron_tagger_eng", download_dir=NLTK_DIR)
nltk.download("maxent_ne_chunker_tab", download_dir=NLTK_DIR)
nltk.download("words", download_dir=NLTK_DIR)

# SpaCy model needed by this notebook
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package words is already up-to-date!


## Data and preprocessing

Load the *SMTeuroparl* sentence pairs with their gold-standard scores, and process every sentence once with spaCy (tokens and NEs). NLTK works directly on the sentence strings. The FreeLing tokens are read from the TextServer answers saved in Lab 6.

In [2]:
# Load the spaCy model (tokens and NEs).
nlp = spacy.load("en_core_web_sm")

# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / "STS.input.SMTeuroparl.txt", sep="\t", header=None)
dt["gs"] = pd.read_csv(DATA_DIR / "STS.gs.SMTeuroparl.txt", sep="\t", header=None)

# Process every sentence once with spaCy.
docs1 = [nlp(s) for s in dt[0]]
docs2 = [nlp(s) for s in dt[1]]

# FreeLing tokens of every sentence, from the TextServer answers saved in Lab 6 (service "senses").
# Each token is [form, lemma, tag, POS, synset], and the words of a NE are joined with "_" (Van_Orden).
ukb_raw = json.loads(UKB_CACHE.read_text())
freeling1 = [ukb_raw[s] for s in dt[0]]
freeling2 = [ukb_raw[s] for s in dt[1]]
print(f"Sentence pairs: {len(dt)}")

Sentence pairs: 459


## Words plus NEs

Each sentence becomes a list of items: the tokens of a NE are merged into a single item and the other tokens are kept.

- **spaCy** gives the NEs as `doc.ents`, spans of tokens with a label (PERSON, ORG, DATE...).
- **NLTK** needs POS tags first: `ne_chunk` returns a tree where each NE is a subtree with its label (PERSON, ORGANIZATION, GPE...).
- **FreeLing** already returns each NE as a single token, with a tag starting with `NP` and its words joined with `_` (`Van_Orden`). It also joins some other multiword expressions (`because_of`, `no_longer`), which are split into their words as the rest.

The example below (a sentence of the corpus) shows the difference: spaCy and FreeLing merge *Van Orden*, while NLTK gives *Van* (PERSON) and *Orden* (ORGANIZATION) as two separate NEs.

In [3]:
# Words plus NEs: the tokens of a NE are merged into a single item, the other tokens are kept.

# spaCy: the NEs are doc.ents (spans of tokens).
def spacy_words(doc):
    return [t.text for t in doc]

def spacy_words_and_nes(doc):
    items, i = [], 0
    starts = {ent.start: ent for ent in doc.ents}
    while i < len(doc):
        if i in starts:
            items.append(starts[i].text)
            i = starts[i].end
        else:
            items.append(doc[i].text)
            i += 1
    return items

# NLTK: the NE chunker returns a tree where each NE is a subtree (PERSON, ORGANIZATION, GPE, ...).
# nltk.ne_chunk loads the chunker from disk at every call, so it is loaded only once here.
ne_chunker = nltk.chunk.ne_chunker()

def nltk_words(sentence):
    return nltk.word_tokenize(sentence)

def nltk_words_and_nes(sentence):
    tree = ne_chunker.parse(nltk.pos_tag(nltk.word_tokenize(sentence)))
    return [" ".join(word for word, tag in node.leaves()) if isinstance(node, nltk.Tree) else node[0]
            for node in tree]

# FreeLing: the NEs are the tokens with a tag NP, the "_" between their words is replaced by a space.
def freeling_words(tokens):
    return [word for form, lemma, tag, pos, wn_id in tokens for word in form.split("_")]

def freeling_words_and_nes(tokens):
    items = []
    for form, lemma, tag, pos, wn_id in tokens:
        if tag.startswith("NP"):
            items.append(form.replace("_", " "))
        else:
            items.extend(form.split("_"))
    return items

# Example: a sentence of the corpus (S2 of pair 393), with the three tools.
example = "Van Orden report (A5-0241/2000)"
print(example)
print("  NLTK:    ", nltk_words_and_nes(example))
print("  spaCy:   ", spacy_words_and_nes(nlp(example)))
print("  FreeLing:", freeling_words_and_nes(ukb_raw[example]))

# The NLTK tree: each word of the name is a separate NE.
print("\nNLTK ne_chunk tree:")
print(ne_chunker.parse(nltk.pos_tag(nltk.word_tokenize(example))))

Van Orden report (A5-0241/2000)
  NLTK:     ['Van', 'Orden', 'report', '(', 'A5-0241/2000', ')']
  spaCy:    ['Van Orden', 'report', '(', 'A5-0241/2000', ')']
  FreeLing: ['Van Orden', 'report', '(', 'A5-0241/2000', ')']

NLTK ne_chunk tree:
(S
  (PERSON Van/NNP)
  (ORGANIZATION Orden/NNP)
  report/NN
  (/(
  A5-0241/2000/NNP
  )/))


## Similarities and Pearson correlation

Jaccard similarity of each pair with words and with words plus NEs, for the three tools, and the Pearson correlation of each approach with the gold standard.

In [4]:
def jaccard(items1, items2):
    return 1 - jaccard_distance(set(items1), set(items2))

dt["spaCy words"] = [jaccard(spacy_words(d1), spacy_words(d2)) for d1, d2 in zip(docs1, docs2)]
dt["spaCy words + NEs"] = [jaccard(spacy_words_and_nes(d1), spacy_words_and_nes(d2)) for d1, d2 in zip(docs1, docs2)]
dt["NLTK words"] = [jaccard(nltk_words(s1), nltk_words(s2)) for s1, s2 in zip(dt[0], dt[1])]
dt["NLTK words + NEs"] = [jaccard(nltk_words_and_nes(s1), nltk_words_and_nes(s2)) for s1, s2 in zip(dt[0], dt[1])]
dt["FreeLing words"] = [jaccard(freeling_words(f1), freeling_words(f2)) for f1, f2 in zip(freeling1, freeling2)]
dt["FreeLing words + NEs"] = [jaccard(freeling_words_and_nes(f1), freeling_words_and_nes(f2)) for f1, f2 in zip(freeling1, freeling2)]

# The vector of similarities, next to the gold standard.
approaches = ["NLTK words", "NLTK words + NEs", "spaCy words", "spaCy words + NEs", "FreeLing words", "FreeLing words + NEs"]
dt[["gs"] + approaches].head().round(4)

,gs,NLTK words,NLTK words + NEs,spaCy words,spaCy words + NEs,FreeLing words,FreeLing words + NEs
0,4.50,0.3462,0.3462,0.3462,0.3462,0.4000,0.4000
1,5.00,0.7857,0.7857,0.7857,0.7857,0.7857,0.7857
2,4.25,0.3913,0.3913,0.3913,0.3913,0.3913,0.3913
3,4.50,0.5455,0.5455,0.6000,0.6667,0.6000,0.6000
4,5.00,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000


In [5]:
# Pearson correlation of each approach with the gold standard.
results = pd.DataFrame({"Pearson": [pearsonr(dt["gs"], dt[a])[0] for a in approaches]}, index=approaches)
results.round(4)

,Pearson
NLTK words,0.4507
NLTK words + NEs,0.4443
spaCy words,0.4606
spaCy words + NEs,0.4351
FreeLing words,0.4507
FreeLing words + NEs,0.4391


## Effect of the NEs, pair by pair

Count the pairs whose similarity changes when the NEs are merged.

In [6]:
# How many pairs are affected by the NEs?
for tool in ["NLTK", "spaCy", "FreeLing"]:
    diff = dt[f"{tool} words + NEs"] - dt[f"{tool} words"]
    print(f"{tool:<8} pairs that change: {(diff != 0).sum():>3}   higher: {(diff > 0).sum():>3}   lower: {(diff < 0).sum():>3}")

NLTK     pairs that change:  35   higher:   0   lower:  35
spaCy    pairs that change:  82   higher:  18   lower:  64
FreeLing pairs that change:  43   higher:   4   lower:  39


## Largest changes

Look at the pairs whose similarity changes most when the NEs are merged, with the three tools, to see where the changes come from.

In [7]:
def show_pair(i):
    print(f"Pair {i}  (gold {dt['gs'][i]})")
    for label, doc in [("S1", docs1[i]), ("S2", docs2[i])]:
        print(f"  {label}: {doc.text}")
        print(f"      NEs: {[(ent.text, ent.label_) for ent in doc.ents]}")
    print(f"  spaCy words {dt['spaCy words'][i]:.3f}   spaCy words + NEs {dt['spaCy words + NEs'][i]:.3f}")
    set1, set2 = set(spacy_words_and_nes(docs1[i])), set(spacy_words_and_nes(docs2[i]))
    print(f"    shared:  {sorted(set1 & set2)}")
    print(f"    only S1: {sorted(set1 - set2)}")
    print(f"    only S2: {sorted(set2 - set1)}")
    print()

diff = dt["spaCy words + NEs"] - dt["spaCy words"]
print("=== Largest decreases with NEs ===\n")
for i in diff.nsmallest(3).index:
    show_pair(i)
print("=== Largest increases with NEs ===\n")
for i in diff.nlargest(2).index:
    show_pair(i)

=== Largest decreases with NEs ===

Pair 393  (gold 5.0)
  S1: Van Orden Report (A5-0241/2000)
      NEs: []
  S2: Van Orden report (A5-0241/2000)
      NEs: [('Van Orden', 'PERSON'), ('A5-0241/2000', 'CARDINAL')]
  spaCy words 0.778   spaCy words + NEs 0.182
    shared:  ['(', ')']
    only S1: ['-', '0241/2000', 'A5', 'Orden', 'Report', 'Van']
    only S2: ['A5-0241/2000', 'Van Orden', 'report']

Pair 31  (gold 4.25)
  S1: Maij-Weggen report (A5-0323/2000)
      NEs: [('Maij-Weggen', 'ORG'), ('A5-0323/2000', 'DATE')]
  S2: Relation Maij-Weggen (A5-0323/2000)
      NEs: [('A5-0323/2000', 'DATE')]
  spaCy words 0.778   spaCy words + NEs 0.333
    shared:  ['(', ')', 'A5-0323/2000']
    only S1: ['Maij-Weggen', 'report']
    only S2: ['-', 'Maij', 'Relation', 'Weggen']

Pair 227  (gold 5.0)
  S1: Maij-Weggen report (A5-0323/2000)
      NEs: [('Maij-Weggen', 'ORG'), ('A5-0323/2000', 'DATE')]
  S2: The report Maij-Weggen, A5-0323/2000 --
      NEs: [('Maij-Weggen', 'ORG'), ('A5', 'CARDINA

In [8]:
# NLTK and FreeLing: the largest decreases with NEs, and pair 393 (the largest decrease with spaCy).
for tool, items, x1s, x2s in [("NLTK", nltk_words_and_nes, dt[0], dt[1]),
                              ("FreeLing", freeling_words_and_nes, freeling1, freeling2)]:
    diff = dt[f"{tool} words + NEs"] - dt[f"{tool} words"]
    print(f"=== {tool}: largest decreases with NEs ===\n")
    for i in list(diff.nsmallest(3).index) + [393]:
        print(f"Pair {i}  (gold {dt['gs'][i]})   {tool} words {dt[f'{tool} words'][i]:.3f}   {tool} words + NEs {dt[f'{tool} words + NEs'][i]:.3f}")
        print(f"  S1 items: {items(x1s[i])}")
        print(f"  S2 items: {items(x2s[i])}")
        print()

=== NLTK: largest decreases with NEs ===

Pair 260  (gold 4.75)   NLTK words 1.000   NLTK words + NEs 0.429
  S1 items: ['Van', 'Orden Report', '(', 'A5-0241/2000', ')']
  S2 items: ['Report', 'Van Orden', '(', 'A5-0241/2000', ')']

Pair 437  (gold 5.0)   NLTK words 0.500   NLTK words + NEs 0.111
  S1 items: ['Van', 'Orden Report', '(', 'A5-0241/2000', ')']
  S2 items: ['Report', 'Van Orden', ',', 'A5-0241/2000', '--']

Pair 90  (gold 5.0)   NLTK words 0.556   NLTK words + NEs 0.200
  S1 items: ['Van', 'Orden Report', '(', 'A5-0241/2000', ')']
  S2 items: ['Report', 'Van Orden', '(', 'A5-0241', '/', '2000', ')']

Pair 393  (gold 5.0)   NLTK words 0.714   NLTK words + NEs 0.571
  S1 items: ['Van', 'Orden Report', '(', 'A5-0241/2000', ')']
  S2 items: ['Van', 'Orden', 'report', '(', 'A5-0241/2000', ')']

=== FreeLing: largest decreases with NEs ===

Pair 260  (gold 4.75)   FreeLing words 1.000   FreeLing words + NEs 0.600
  S1 items: ['Van Orden Report', '(', 'A5-0241/2000', ')']
  S2 it

## Question

### Results

Each sentence is represented as its set of words, where the words of a NE are merged into a single item, and the pairs are compared with the Jaccard coefficient. The NEs are found with NLTK (`ne_chunk`), spaCy (`en_core_web_sm`) and FreeLing (TextServer, the answers saved in Lab 6), and each one is compared with the words of the same tokenizer without NEs:

| Approach | Pearson |
|---|---|
| NLTK words (Lab 2) | 0.4507 |
| NLTK words + NEs | 0.4443 |
| spaCy words | 0.4606 |
| spaCy words + NEs | 0.4351 |
| FreeLing words | 0.4507 |
| FreeLing words + NEs | 0.4391 |

**Adding the NEs lowers the correlation with the three tools**: -0.0064 with NLTK, -0.0255 with spaCy and -0.0116 with FreeLing. With NLTK, the 35 pairs that change all get a lower similarity. With spaCy, 82 pairs change: 64 get lower and only 18 get higher. With FreeLing, 43 pairs change: 39 get lower and 4 get higher.

### Do you think it could be relevant to use NEs to compute the similarity between two sentences?

**In principle yes, but not in this way and not on this dataset.**

**Why NEs could be relevant.** NEs carry a big part of the meaning of a sentence: two sentences about *Tunisia* and *Latvia*, or about the *Van Orden* and the *Maij-Weggen* reports, are about different things even if all their other words are the same. Keeping a NE as a single item also avoids partial matches, e.g. *John Smith* and *John Brown* would no longer share the item *John*.

**Why it does not help here.**

- **Merging only helps if both sentences are analysed in the same way.** A multiword NE is a single item, so if it is recognized in one sentence but not in the other, its words no longer match at all. In pair 393 (gold 5.0), *"Van Orden Report (A5-0241/2000)"* has no NE but *"Van Orden report (A5-0241/2000)"* has `Van Orden` (PERSON), so the similarity drops from 0.778 to 0.182. The only difference between the two sentences is the case of *report*, which changes the analysis of the whole sentence. FreeLing has the same problem in this pair: *Van Orden Report* is one NE in S1, but in S2 the NE is only *Van Orden* (0.714 → 0.500). In pair 31 (gold 4.25), *Maij-Weggen* is a NE in S1 but not in S2, where it stays as *Maij*, *-*, *Weggen* (0.778 → 0.333).
- **The same words in a different order give a different NE.** With FreeLing, the largest decreases all come from the *Van Orden* report: in pair 260 (gold 4.75), *Van Orden Report* and *Report Van Orden* have exactly the same words, but they are two different NEs, so the similarity drops from 1.000 to 0.600. Pairs 420 and 437 (gold 4.5 and 5.0) lose half of their similarity in the same way.
- **The NE span is not the same in both sentences.** In pair 227 (gold 5.0), *A5-0323/2000* is one NE in S1, but only *A5* is a NE in S2, so the reference number does not match any more.
- **NLTK merges very few words.** Its chunker often gives each word of a name its own NE, so they stay separate: in *Van Orden report (A5-0241/2000)*, *Van* is a PERSON and *Orden* an ORGANIZATION, so the output is the same as without NEs, while spaCy and FreeLing merge *Van Orden*. Only 35 pairs change, and all of them get worse. When it does merge words, the span is often wrong: in pair 260 (gold 4.75), S1 becomes *Van* + *Orden Report* and S2 *Report* + *Van Orden*, so none of the NEs match and the similarity drops from 1.000 to 0.429, the largest decrease of the three tools.
- **The increases are not real NE matches.** The two largest increases with spaCy (pairs 122 and 3) come from NEs that are *different* in the two sentences: merging them only reduces the number of non-shared items. Pair 122 even has a low gold score (3.0).
- **The SMTeuroparl pairs share most of their NEs anyway.** The second sentence is a translation of the first, so the NEs are usually the same in both, and word overlap already captures them.

**Conclusion.** The three tools give the same result, so the problem is not one particular recognizer but the approach. NEs are useful information for sentence similarity, but they should be used in a way that does not depend on getting exactly the same span in both sentences: for example normalizing the NEs (e.g. lowercase), comparing the NEs as an additional feature instead of replacing the words, or using the NE types (e.g. both sentences mention a PERSON and a DATE). Used as merged tokens, the errors and inconsistencies of the recognizer cost more than the information the NEs add.